# HW4: Evaluation

LLM Zoomcamp 2026 — Module 4

## Setup

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel
from tqdm.auto import tqdm
from openai import OpenAI

load_dotenv("/Users/dt00035/projects/personal/reference/llm-zoomcamp/.env")

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

## Q1. Generating questions — average input tokens

In [ ]:
class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

# Using Groq (OpenAI-compatible) since no OpenAI key available
groq_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

TARGET_PAGES = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

first_3 = sorted(
    [d for d in documents if d["filename"] in TARGET_PAGES],
    key=lambda d: TARGET_PAGES.index(d["filename"])
)

token_counts = []
for doc in first_3:
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": data_gen_instructions + "\n\nRespond with JSON only."},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
    )
    token_counts.append(response.usage.prompt_tokens)
    print(f"{doc['filename']}: {response.usage.prompt_tokens} input tokens")

avg_tokens = sum(token_counts) / len(token_counts)
print(f"\nAverage input tokens: {avg_tokens:.0f}")

## Build search indices

In [ ]:
from minsearch import Index
from minsearch.vector import VectorSearch
from embedder import Embedder

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks")

txt_index = Index(text_fields=["content"], keyword_fields=["filename", "start"])
txt_index.fit(chunks)

embedder = Embedder()
X = embedder.encode_batch([c["content"] for c in chunks])
print(f"Embeddings: {X.shape}")

vec_index = VectorSearch(keyword_fields=["filename", "start"])
vec_index.fit(X, chunks)

def text_search(query, num_results=5):
    return txt_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    return vec_index.search(embedder.encode(query), num_results=num_results)

def rrf(result_lists, k=60, num_results=5):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    return rrf([text_search(query, 10), vector_search(query, 10)], k=k, num_results=num_results)

In [ ]:
df = pd.read_csv("ground-truth.csv")
ground_truth = df.to_dict(orient="records")
print(f"Loaded {len(ground_truth)} ground truth records")
print(f"First question: {ground_truth[0]['question']}")

## Q2. First result with text_search

In [ ]:
q = ground_truth[0]["question"]
print(f"Query: {q}")
print(f"Expected: {ground_truth[0]['filename']}")

text_results = text_search(q)
print(f"\nQ2 Answer: {text_results[0]['filename']}")
for r in text_results:
    print(f"  {r['filename']}'")

## Q3. First result with vector_search

In [ ]:
vec_results = vector_search(q)
print(f"Q3 Answer: {vec_results[0]['filename']}")
for r in vec_results:
    print(f"  {r['filename']}")

## Evaluation helpers

In [ ]:
def compute_relevance(q_record, search_fn):
    filename = q_record["filename"]
    results = search_fn(q_record["question"])
    return [int(r["filename"] == filename) for r in results]

def hit_rate(relevance_total):
    return sum(1 for r in relevance_total if 1 in r) / len(relevance_total)

def mrr(relevance_total):
    total = 0.0
    for r in relevance_total:
        for rank, val in enumerate(r):
            if val == 1:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)

def evaluate(ground_truth, search_fn, desc=""):
    relevance_total = [compute_relevance(q, search_fn) for q in tqdm(ground_truth, desc=desc)]
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}

## Q4. text_search Hit Rate

In [ ]:
text_eval = evaluate(ground_truth, text_search, "text_search")
print(f"Hit Rate: {text_eval['hit_rate']:.4f}")
print(f"MRR:      {text_eval['mrr']:.4f}")

## Q5. vector_search MRR

In [ ]:
vec_eval = evaluate(ground_truth, vector_search, "vector_search")
print(f"Hit Rate: {vec_eval['hit_rate']:.4f}")
print(f"MRR:      {vec_eval['mrr']:.4f}")

## Q6. Hybrid search — best k

In [ ]:
hybrid_results = {}
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth, lambda q, k=k: hybrid_search(q, k=k), f"hybrid k={k}")
    hybrid_results[k] = result
    print(f"k={k:3d}: MRR={result['mrr']:.4f}, Hit Rate={result['hit_rate']:.4f}")

max_mrr = max(r["mrr"] for r in hybrid_results.values())
tied_ks = [k for k in [1, 50, 100, 200] if hybrid_results[k]["mrr"] == max_mrr]
best_k = min(tied_ks)
print(f"\nBest k = {best_k} (MRR={max_mrr:.4f})")

## Summary

In [ ]:
print("=" * 50)
print("FINAL ANSWERS")
print("=" * 50)
print(f"Q1: ~1400 tokens (actual avg: {avg_tokens:.0f})")
print(f"Q2: {text_results[0]['filename']}")
print(f"Q3: {vec_results[0]['filename']}")
print(f"Q4: Hit Rate = {text_eval['hit_rate']:.4f}  → select 0.76")
print(f"Q5: MRR = {vec_eval['mrr']:.4f}  → select 0.55")
print(f"Q6: Best k = {best_k}")